# BONUS 2: Image Retrieval (2 балла)

## Внимание! За бонусы доп. баллы не ставятся, но вы можете сделать их для себя.

Давайте представим, что весь наш тренировочный датасет -- это большая база данных людей. И вот мы получили картинку лица какого-то человека с уличной камеры наблюдения (у нас это картинка из тестового датасета) и хотим понять, что это за человек. Что нам делать? Правильно -- берем наш VAE, кодируем картинку в латентное представление и ищем среди латентных представлений лиц нашей базы самые ближайшие!

План:

1. Получаем латентные представления всех лиц тренировочного датасета
2. Обучаем на них LSHForest `(sklearn.neighbors.LSHForest)`, например, с `n_estimators=50`
3. Берем картинку из тестового датасета, с помощью VAE получаем ее латентный вектор
4. Ищем с помощью обученного LSHForest ближайшие из латентных представлений тренировочной базы
5. Находим лица тренировочного датасета, которым соответствуют ближайшие латентные представления, визуализируем!

Немного кода вам в помощь: (feel free to delete everything and write your own)

In [1]:
import os

import imageio
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import datasets
from torchvision import transforms
import torch.nn.functional as F



%matplotlib inline

In [2]:
from pathlib import Path

In [3]:
# Скачиваем картинки
images_path = kagglehub.dataset_download("jessicali9530/lfw-dataset")
print("Path to dataset files:", images_path)

Path to dataset files: C:\Users\k142\.cache\kagglehub\datasets\jessicali9530\lfw-dataset\versions\4


In [4]:
# Скачиваем атрибуты
attrs_path = kagglehub.dataset_download("averkij/lfw-attributes")
print("Path to dataset files:", attrs_path)

Path to dataset files: C:\Users\k142\.cache\kagglehub\datasets\averkij\lfw-attributes\versions\1


In [5]:
# DATASET_PATH ="/kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled/"
# ATTRIBUTES_PATH = "/kaggle/input/lfw-attributes/lfw_attributes.txt"
DATASET_PATH = Path(images_path)
ATTRIBUTES_PATH = Path(attrs_path).joinpath("lfw_attributes.txt")

In [6]:
def fetch_dataset(dx=80, dy=80, dimx=45, dimy=45):
    df_attrs = pd.read_csv(ATTRIBUTES_PATH, sep='\t', skiprows=1, )
    df_attrs = pd.DataFrame(df_attrs.iloc[:, :-1].values, columns=df_attrs.columns[1:])

    photo_ids = []
    for dirpath, dirnames, filenames in os.walk(DATASET_PATH):
        for fname in filenames:
            if fname.endswith(".jpg"):
                fpath = os.path.join(dirpath, fname)
                photo_id = fname[:-4].replace('_', ' ').split()
                person_id = ' '.join(photo_id[:-1])
                photo_number = int(photo_id[-1])
                photo_ids.append({'person': person_id, 'imagenum': photo_number, 'photo_path': fpath})

    photo_ids = pd.DataFrame(photo_ids)
    # print(photo_ids.info())
    df = pd.merge(df_attrs, photo_ids, on=('person', 'imagenum'))

    assert len(df) == len(df_attrs), "Потеряны данные при объединении датафреймов!"

    # Обрезка
    # dy - количество пикселей, обрезаемых сверху и снизу
    # dx - количество пикселей, обрезаемых слева и справа
    #
    # Изменение размера
    # dimx - ширина
    # dimy - высота

    images = df['photo_path'].apply(imageio.imread) \
        .apply(lambda img: img[dy:-dy, dx:-dx]) \
        .apply(lambda img: np.array(Image.fromarray(img).resize([dimx, dimy])))

    images = np.stack(images.values).astype('uint8')
    attrs = df.drop(["photo_path", "person", "imagenum"], axis=1)

    return images, attrs

In [7]:
# Обратите внимание, что датасет представляет собой не только картинки, но и атрибуты
# Атрибуты понадобятся в конце этого задания

images, attrs = fetch_dataset(dimx=44, dimy=44)

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torchvision import transforms

class AutoencoderDataset(Dataset):
    def __init__(self, images, indices, transform=None):
        """
        Args:
            images: numpy array в формате (N, H, W, C)
            indices: индексы для выборки
            transform: преобразования (включая ToTensor())
        """
        self.indices = indices
        self.images = images[indices]  # Выбираем данные по индексам (остаётся H x W x C)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]  # Получаем изображение в формате H x W x C

        if self.transform:
            image = self.transform(image)  # Применяем преобразования (включая ToTensor())
        else:
            # Если transform=None, делаем минимальное преобразование в тензор
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0  # HWC -> CHW и [0,255] -> [0,1]

        return image, image  # Для автоэнкодера input и target одинаковы

def get_dataloaders(images, batch_size=32, train_ratio=0.8, val_ratio=0.1):
    """
    Создает DataLoader'ы для train, validation и test

    Args:
        images: numpy array в формате (N, H, W, C)
        batch_size: размер батча
        train_ratio: доля тренировочных данных
        val_ratio: доля валидационных данных
    Returns:
        train_loader, val_loader, test_loader
    """
    # Проверка формата
    if images.ndim != 4 or images.shape[3] not in {1, 3}:
        raise ValueError("Ожидается формат (N, H, W, C), где C=1 или 3")

    # Разделение данных
    n_total = len(images)
    ix = np.random.choice(n_total, n_total, False)  # Перемешанные индексы
    train_size = int(train_ratio * n_total)
    val_size = int(val_ratio * n_total)
    tr, val, ts = np.split(ix, [train_size, train_size + val_size])

    # Стандартные преобразования (HWC -> CHW + нормализация)
    transform = transforms.Compose([
        transforms.ToTensor()  # Конвертирует numpy (H x W x C) в torch (C x H x W) и нормализует [0,1]
    ])

    # Создаем датасеты
    train_dataset = AutoencoderDataset(images, tr, transform=transform)
    val_dataset = AutoencoderDataset(images, val, transform=transform)
    test_dataset = AutoencoderDataset(images, ts, transform=transform)

    # Создаем DataLoader'ы
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [9]:
train_loader, val_loader, test_loader = get_dataloaders(images, batch_size=64)

In [105]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VariationalEncoder(nn.Module):
    def __init__(self, latent_dims):
        super(VariationalEncoder, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1)   # 64x64 → 32x32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1)  # 32x32 → 16x16
        self.conv3 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1) # 16x16 → 8x8
        self.conv4 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1) # 8x8 → 4x4

        self.flatten = nn.Flatten()
        # self.fc_mu = nn.Linear(256 * 4 * 4, latent_dims)
        # self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dims)
        self.fc_mu = nn.Linear(256 * 2 * 2, latent_dims)  # или другой размер
        self.fc_logvar = nn.Linear(256 * 2 * 2, latent_dims)
        self.N = torch.distributions.Normal(0, 1)
        self.kl = 0

        self.test_out_dim = None  # для отладки


    def forward(self, x):
        x = F.relu(self.conv1(x))  # [B, 3, 64, 64] → [B, 32, 32, 32]
        x = F.relu(self.conv2(x))  # → [B, 64, 16, 16]
        x = F.relu(self.conv3(x))  # → [B, 128, 8, 8]
        x = F.relu(self.conv4(x))  # → [B, 256, 4, 4]
        if self.test_out_dim is None:
            print("Feature map size:", x.shape)  # [B, C, H, W]
            self.test_out_dim = x.shape[1] * x.shape[2] * x.shape[3]
        x = self.flatten(x)        # → [B, 4096]

        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        std = torch.exp(0.5 * logvar)
        z = mu + std * self.N.sample(mu.shape).to(x.device)

        self.kl = -0.5 * torch.sum(1 + logvar - mu**2 - logvar.exp())
        return z


In [106]:
class Decoder(nn.Module):
    def __init__(self, latent_dims):
        super(Decoder, self).__init__()
        self.fc = nn.Linear(latent_dims, 256 * 2 * 2)

        self.deconv1 = nn.ConvTranspose2d(256, 128, 4, 2, 1)  # 2→4
        self.deconv2 = nn.ConvTranspose2d(128, 64, 4, 2, 1)   # 4→8
        self.deconv3 = nn.ConvTranspose2d(64, 64, 4, 2, 1)    # 8→16
        self.deconv4 = nn.ConvTranspose2d(64, 32, 4, 2, 1)    # 16→32
        self.deconv5 = nn.ConvTranspose2d(32, 3, 4, 2, 1)     # 32→64

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 256, 2, 2)

        x = F.relu(self.deconv1(x))   # 2→4
        x = F.relu(self.deconv2(x))   # 4→8
        x = F.relu(self.deconv3(x))   # 8→16
        x = F.relu(self.deconv4(x))   # 16→32
        x = torch.sigmoid(self.deconv5(x))  # 32→64

        return x


In [107]:
class VariationalAutoencoder(nn.Module):
    def __init__(self, latent_dims):
        super(VariationalAutoencoder, self).__init__()
        self.encoder = VariationalEncoder(latent_dims)
        self.decoder = Decoder(latent_dims)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


In [108]:
def train(autoencoder, data, epochs=20):
    opt = torch.optim.Adam(autoencoder.parameters())
    autoencoder.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        num_batches = 0

        for x, _ in data:
            x = x.to(device)
            opt.zero_grad()
            x_hat = autoencoder(x)
            recon_loss = ((x - x_hat) ** 2).sum()
            loss = recon_loss + autoencoder.encoder.kl
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            num_batches += 1

        print(f'Epoch: {epoch}, Loss: {epoch_loss / num_batches:.4f}')


### Обучение VAE

In [109]:
latent_dims = 64

In [110]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [111]:
vae = VariationalAutoencoder(latent_dims).to(device) # GPU
vae = train(vae, train_loader,epochs=20)

Feature map size: torch.Size([64, 256, 2, 2])


RuntimeError: The size of tensor a (44) must match the size of tensor b (64) at non-singleton dimension 3

In [ ]:
codes = <поучите латентные представления картинок из трейна>

In [ ]:
# обучаем LSHForest
from sklearn.neighbors import LSHForest
lshf = LSHForest(n_estimators=50).fit(codes)

In [ ]:
def get_similar(image, n_neighbors=5):
  # функция, которая берет тестовый image и с помощью метода kneighbours у LSHForest ищет ближайшие векторы
  # прогоняет векторы через декодер и получает картинки ближайших людей

  code = <получение латентного представления image>

  (distances,),(idx,) = lshf.kneighbors(code, n_neighbors=n_neighbors)

  return distances, X_train[idx]

In [ ]:
def show_similar(image):

  # функция, которая принимает тестовый image, ищет ближайшие к нему и визуализирует результат

    distances,neighbors = get_similar(image,n_neighbors=11)

    plt.figure(figsize=[8,6])
    plt.subplot(3,4,1)
    plt.imshow(image.cpu().numpy().transpose([1,2,0]))
    plt.title("Original image")

    for i in range(11):
        plt.subplot(3,4,i+2)
        plt.imshow(neighbors[i].cpu().numpy().transpose([1,2,0]))
        plt.title("Dist=%.3f"%distances[i])
    plt.show()

In [ ]:
<тут выведите самые похожие лица к какому-нибудь лицу из тестовой части датасета>